In [ ]:
import sys
import subprocess

required_packages = [
    "sentence-transformers",
    "faiss-cpu",
    "langchain",
    "langchain-text-splitters",
    "transformers",
    "PyPDF2",
    "pdfplumber",
    "pandas",
    "openpyxl",
    "rouge-score",
    "bert-score",
    "nltk",
    "evaluate",
    "pdfplumber"
]

PACKAGE_IMPORT_MAP = {
    "sentence-transformers":    "sentence_transformers",
    "faiss-cpu":                "faiss",
    "langchain-text-splitters": "langchain_text_splitters",
    "rouge-score":              "rouge_score",
    "bert-score":               "bert_score",
    "PyPDF2":                   "PyPDF2",
}

for package in required_packages:
    import_name = PACKAGE_IMPORT_MAP.get(package, package.replace("-", "_"))
    try:
        __import__(import_name)
        print(f" {package} already installed")
    except ImportError:
        print(f"Installing {package}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])
        print(f" {package} installed")

 sentence-transformers already installed
Installing faiss-cpu...
 faiss-cpu installed
 langchain already installed
Installing langchain-text-splitters...
 langchain-text-splitters installed
 transformers already installed
Installing PyPDF2...
 PyPDF2 installed
Installing pdfplumber...
 pdfplumber installed
 pandas already installed
 openpyxl already installed
Installing rouge-score...
 rouge-score installed
Installing bert-score...
 bert-score installed
 nltk already installed
Installing evaluate...
 evaluate installed
 pdfplumber already installed


In [ ]:
import random
import numpy as np
import torch
import os
import json
import pickle
import warnings
import re
import io
import pdfplumber
import gc
import itertools
from typing import List, Dict, Tuple, Optional, Any
from dataclasses import dataclass
from datetime import datetime
import pandas as pd
from PyPDF2 import PdfReader
import nltk
from nltk.tokenize import sent_tokenize

for resource in ["punkt", "punkt_tab", "wordnet", "omw-1.4"]:
    try:
        if resource in ("punkt", "punkt_tab"):
            nltk.data.find(f"tokenizers/{resource}")
        else:
            nltk.data.find(f"corpora/{resource}")
    except LookupError:
        nltk.download(resource, quiet=True)

from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer, CrossEncoder
import faiss
import pdfplumber
from collections import Counter
from transformers import AutoTokenizer, AutoModelForCausalLM
from tqdm import tqdm
from rouge_score import rouge_scorer
from bert_score import score as bert_score
import evaluate
from google.colab import drive, auth
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload

warnings.filterwarnings("ignore")

print(" All libraries imported")

 All libraries imported


In [ ]:
import importlib

print(f"{'Package':<30} {'Version'}")
print("-" * 50)

packages = {
    "torch":                    "torch",
    "transformers":             "transformers",
    "sentence_transformers":    "sentence-transformers",
    "faiss":                    "faiss-cpu",
    "accelerate":               "accelerate",
    "huggingface_hub":          "huggingface-hub",
    "langchain_text_splitters": "langchain-text-splitters",
    "PyPDF2":                   "PyPDF2",
    "rouge_score":              "rouge-score",
    "bert_score":               "bert-score",
    "evaluate":                 "evaluate",
    "nltk":                     "nltk",
    "pandas":                   "pandas",
    "openpyxl":                 "openpyxl",
    "numpy":                    "numpy",
    "tqdm":                     "tqdm",
}

for import_name, pip_name in packages.items():
    try:
        mod = importlib.import_module(import_name)
        version = getattr(mod, "__version__", None)
        if version is None:
            import importlib.metadata
            version = importlib.metadata.version(pip_name)
        print(f"  {pip_name:<28} {version}")
    except Exception:
        print(f"  {pip_name:<28} NOT FOUND")

print("-" * 50)
print(f"  {'Python':<28} {sys.version.split()[0]}")
print(f"  {'CUDA':<28} {torch.version.cuda}")
print(f"  {'GPU':<28} {torch.cuda.get_device_name(0)}")
print(f"  {'VRAM':<28} {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

Package                        Version
--------------------------------------------------
  torch                        2.10.0+cu128
  transformers                 5.0.0
  sentence-transformers        5.3.0
  faiss-cpu                    1.13.2
  accelerate                   1.13.0
  huggingface-hub              1.7.1
  langchain-text-splitters     1.1.1
  PyPDF2                       3.0.1
  rouge-score                  0.1.2
  bert-score                   0.3.12
  evaluate                     0.4.6
  nltk                         3.9.1
  pandas                       2.2.2
  openpyxl                     3.1.5
  numpy                        2.0.2
  tqdm                         4.67.3
--------------------------------------------------
  Python                       3.12.13
  CUDA                         12.8
  GPU                          NVIDIA RTX PRO 6000 Blackwell Server Edition
  VRAM                         102.0 GB


In [ ]:
RANDOM_SEED = 42

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

print(f" Random seed set to {RANDOM_SEED}\n")

 Random seed set to 42



In [ ]:
DOC_FOLDER          = "/content/drive/MyDrive/Knee_Arthroplasty_RAG/Data"
CACHE_FOLDER        = "/content/drive/MyDrive/Knee_Arthroplasty_RAG/Cache"
EXPERIMENT_LOG_PATH = "/content/drive/MyDrive/Knee_Arthroplasty_RAG/final_experiments_optimized.xlsx"
TUNED_PARAMS_PATH   = "/content/drive/MyDrive/Knee_Arthroplasty_RAG/best_parameters_optimized.json"

EMBEDDING_MODEL_NAME = "BAAI/bge-m3"
LLM_MODEL_NAME       = "google/medgemma-27b-it"

print(f"  Documents      : {DOC_FOLDER}")
print(f"  Cache          : {CACHE_FOLDER}")
print(f"  Results        : {EXPERIMENT_LOG_PATH}")
print(f"  Tuned params   : {TUNED_PARAMS_PATH}")
print(f"  Embedding Model: {EMBEDDING_MODEL_NAME}")
print(f"  LLM Model      : {LLM_MODEL_NAME}\n")

  Documents      : /content/drive/MyDrive/Knee_Arthroplasty_RAG/Data
  Cache          : /content/drive/MyDrive/Knee_Arthroplasty_RAG/Cache
  Results        : /content/drive/MyDrive/Knee_Arthroplasty_RAG/final_experiments_optimized.xlsx
  Tuned params   : /content/drive/MyDrive/Knee_Arthroplasty_RAG/best_parameters_optimized.json
  Embedding Model: BAAI/bge-m3
  LLM Model      : google/medgemma-27b-it



In [ ]:
doc_metadata: Dict[str, Dict[str, Any]] = {
    "2018 Insall and Scott Surgery of the Knee 6e.pdf": {
        "book_title": "Insall & Scott Surgery of the Knee",
        "author": "W. Norman Scott",
        "year": 2018,
    },
    "aaos guidlines .pdf": {
        "book_title": "Surgical Management of Osteoarthritis of the Knee",
        "author": "AAOS",
        "year": 2022,
    },
    "Campbell's Operative Orthopaedics 14th Edition.pdf": {
        "book_title": "Campbell's Operative Orthopaedics",
        "author": "Frederick M. Azar, James H. Beaty",
        "year": 2021,
    },
    "Master_Techniques_in_orthopaedic_surgery_knee_Arthroplasty.pdf": {
        "book_title": "Master Techniques in Orthopaedic Surgery: Knee Arthroplasty",
        "author": "Mark Pagnano, Arlen Hanssen",
        "year": 2019,
    },
    "Noyes_Knee_Disorders_Surgery_Rehabilitation.pdf": {
        "book_title": "Noyes' Knee Disorders: Surgery, Rehabilitation, Clinical Outcomes",
        "author": "Frank R. Noyes",
        "year": 2017,
    },
    "Partial knee arthroplasty;techniques for optimal outcomes.pdf": {
        "book_title": "Partial Knee Arthroplasty: Techniques for Optimal Outcomes",
        "author": "Keith R. Berend, Fred D. Cushner",
        "year": 2022,
    },
    "Total Knee Arthroplasty-Richard D Scott 2nd.pdf": {
        "book_title": "Total Knee Arthroplasty",
        "author": "Richard D. Scott",
        "year": 2015,
    },
    "Unicompartmental_Knee_Arthroplasty.pdf": {
        "book_title": "Unicompartmental Knee Arthroplasty",
        "author": "Tad L. Gerlinger",
        "year": 2020,
    },
}

print(" Metadata configured")

 Metadata configured


In [ ]:
@dataclass
class PageSource:
    document_name: str
    page_number: int
    book_title: str
    author: Optional[str] = None
    publication_year: Optional[int] = None

    def get_citation(self) -> str:
        parts = []
        if self.author:
            parts.append(self.author.strip())
        if self.book_title:
            parts.append(self.book_title.strip())
        else:
            parts.append(self.document_name)
        if self.publication_year:
            parts.append(f"({self.publication_year})")
        citation = ". ".join(parts)
        citation += f". Page {max(1, self.page_number)}"
        return citation

print("PageSource defined")

PageSource defined


In [ ]:
class ExperimentLogger:
    def __init__(self, excel_path: str, clear_existing: bool = False):
        self.excel_path = excel_path
        self.sheet_name = "Experiment_Results"
        if clear_existing and os.path.exists(self.excel_path):
            try:
                os.remove(self.excel_path)
                print(f" Existing log file '{os.path.basename(self.excel_path)}' cleared.")
            except Exception as e:
                print(f"Warning: Could not clear existing log file. Error: {e}")
        self._create_if_not_exists()

    def _create_if_not_exists(self):
        if not os.path.exists(self.excel_path):
            os.makedirs(os.path.dirname(self.excel_path), exist_ok=True)
            headers = pd.DataFrame(columns=[
                "Timestamp", "Experiment_ID", "Query", "Configuration",
                "Ground_Truth", "Has_Ground_Truth", "Answer",
                "Answer_Length_Words", "Answer_Length_Chars",
                "Citations", "Citation_Count",
                "FAISS_Top_Similarity", "FAISS_Avg_Similarity",
                "FAISS_Min_Similarity",
                "CrossEncoder_Top_Score", "CrossEncoder_Avg_Score",
                "CrossEncoder_Min_Score", "CrossEncoder_Max_Score",
                "Total_Retrieved_Chunks", "Reranking_Applied",
                "ROUGE1_Precision", "ROUGE1_Recall", "ROUGE1_F1",
                "ROUGE2_Precision", "ROUGE2_Recall", "ROUGE2_F1",
                "ROUGEL_Precision", "ROUGEL_Recall", "ROUGEL_F1",
                "BERTScore_Precision", "BERTScore_Recall", "BERTScore_F1",
                "METEOR_Score", "chunk_size", "chunk_overlap",
                "top_k_retrieval", "rerank_enabled", "temperature",
                "embedding_model", "llm_model",
                "execution_time_seconds", "status", "error_message"
            ])
            with pd.ExcelWriter(self.excel_path, engine="openpyxl") as writer:
                headers.to_excel(writer, sheet_name=self.sheet_name, index=False)

    def log_experiment(
        self,
        query: str,
        answer: str,
        configuration: str,
        ground_truth: Optional[str] = None,
        rouge_scores: Optional[Dict[str, Dict[str, float]]] = None,
        bertscore_metrics: Optional[Dict[str, float]] = None,
        meteor_score: Optional[float] = None,
        retrieval_metrics: Optional[Dict[str, Any]] = None,
        citations: Optional[List[str]] = None,
        hyperparameters: Optional[Dict[str, Any]] = None,
        execution_time: float = 0.0,
        status: str = "success",
        error_message: str = "",
        experiment_id: Optional[str] = None
    ):
        if experiment_id is None:
            experiment_id = f"EXP_{datetime.now().strftime('%Y%m%d_%H%M%S_%f')}"

        if rouge_scores is None:
            rouge_scores = {
                'rouge1': {'precision': 0, 'recall': 0, 'fmeasure': 0},
                'rouge2': {'precision': 0, 'recall': 0, 'fmeasure': 0},
                'rougeL': {'precision': 0, 'recall': 0, 'fmeasure': 0}
            }
        if bertscore_metrics is None:
            bertscore_metrics = {'precision': 0, 'recall': 0, 'f1': 0}
        if retrieval_metrics is None:
            retrieval_metrics = {}
        if citations is None:
            citations = []
        if hyperparameters is None:
            hyperparameters = {}

        row_data = {
            "Timestamp": datetime.now(),
            "Experiment_ID": experiment_id,
            "Query": query[:1000] if query else "",
            "Configuration": configuration,
            "Ground_Truth": ground_truth[:5000] if ground_truth else "",
            "Has_Ground_Truth": 1 if ground_truth else 0,
            "Answer": answer[:5000] if answer else "",
            "Answer_Length_Words": len(answer.split()) if answer else 0,
            "Answer_Length_Chars": len(answer) if answer else 0,
            "Citations": " | ".join(citations) if citations else "",
            "Citation_Count": len(citations),
            "FAISS_Top_Similarity": retrieval_metrics.get("faiss_top_similarity", 0),
            "FAISS_Avg_Similarity": retrieval_metrics.get("faiss_avg_similarity", 0),
            "FAISS_Min_Similarity": retrieval_metrics.get("faiss_min_similarity", 0),
            "CrossEncoder_Top_Score": retrieval_metrics.get("ce_top_score", 0),
            "CrossEncoder_Avg_Score": retrieval_metrics.get("ce_avg_score", 0),
            "CrossEncoder_Min_Score": retrieval_metrics.get("ce_min_score", 0),
            "CrossEncoder_Max_Score": retrieval_metrics.get("ce_max_score", 0),
            "Total_Retrieved_Chunks": retrieval_metrics.get("total_chunks", 0),
            "Reranking_Applied": retrieval_metrics.get("reranking_applied", False),
            "ROUGE1_Precision": rouge_scores.get('rouge1', {}).get('precision', 0),
            "ROUGE1_Recall": rouge_scores.get('rouge1', {}).get('recall', 0),
            "ROUGE1_F1": rouge_scores.get('rouge1', {}).get('fmeasure', 0),
            "ROUGE2_Precision": rouge_scores.get('rouge2', {}).get('precision', 0),
            "ROUGE2_Recall": rouge_scores.get('rouge2', {}).get('recall', 0),
            "ROUGE2_F1": rouge_scores.get('rouge2', {}).get('fmeasure', 0),
            "ROUGEL_Precision": rouge_scores.get('rougeL', {}).get('precision', 0),
            "ROUGEL_Recall": rouge_scores.get('rougeL', {}).get('recall', 0),
            "ROUGEL_F1": rouge_scores.get('rougeL', {}).get('fmeasure', 0),
            "BERTScore_Precision": bertscore_metrics.get('precision', 0),
            "BERTScore_Recall": bertscore_metrics.get('recall', 0),
            "BERTScore_F1": bertscore_metrics.get('f1', 0),
            "METEOR_Score": meteor_score if meteor_score is not None else 0,
            "chunk_size": hyperparameters.get('chunk_size', ""),
            "chunk_overlap": hyperparameters.get('chunk_overlap', ""),
            "top_k_retrieval": hyperparameters.get('top_k', ""),
            "rerank_enabled": hyperparameters.get('rerank_enabled', ""),
            "temperature": hyperparameters.get('temperature', ""),
            "embedding_model": hyperparameters.get('embedding_model', ""),
            "llm_model": hyperparameters.get('llm_model', ""),
            "execution_time_seconds": round(execution_time, 3),
            "status": status,
            "error_message": error_message[:200] if error_message else ""
        }

        try:
            if os.path.exists(self.excel_path):
                existing_df = pd.read_excel(self.excel_path, sheet_name=self.sheet_name)
                next_row = len(existing_df) + 1
                with pd.ExcelWriter(
                    self.excel_path,
                    engine="openpyxl",
                    mode="a",
                    if_sheet_exists="overlay"
                ) as writer:
                    pd.DataFrame([row_data]).to_excel(
                        writer,
                        sheet_name=self.sheet_name,
                        index=False,
                        header=False,
                        startrow=next_row
                    )
            else:
                pd.DataFrame([row_data]).to_excel(
                    self.excel_path,
                    sheet_name=self.sheet_name,
                    index=False
                )
        except Exception as e:
            print(f"Error logging experiment: {e}")

print(" ExperimentLogger defined")

 ExperimentLogger defined


In [ ]:
class CacheManager:
    def __init__(self, cache_folder: str):
        self.cache_folder = cache_folder
        os.makedirs(cache_folder, exist_ok=True)

    def _get_chunks_path(self, chunk_size: int, chunk_overlap: int) -> str:
        return os.path.join(
            self.cache_folder,
            f"chunks_size{chunk_size}_overlap{chunk_overlap}.pkl"
        )

    def _get_embeddings_path(self, chunk_size: int, chunk_overlap: int, model_name: str) -> str:
        model_short = model_name.split('/')[-1]
        return os.path.join(
            self.cache_folder,
            f"embeddings_size{chunk_size}_overlap{chunk_overlap}_{model_short}.pkl"
        )

    def chunks_exist(self, chunk_size: int, chunk_overlap: int) -> bool:
        path = self._get_chunks_path(chunk_size, chunk_overlap)
        exists = os.path.exists(path)
        if exists:
            print(f" Found cached chunks: {os.path.basename(path)}")
        return exists

    def embeddings_exist(self, chunk_size: int, chunk_overlap: int, model_name: str) -> bool:
        path = self._get_embeddings_path(chunk_size, chunk_overlap, model_name)
        exists = os.path.exists(path)
        if exists:
            print(f" Found cached embeddings: {os.path.basename(path)}")
        return exists

    def save_chunks(self, chunks: List[Dict], chunk_size: int, chunk_overlap: int):
        path = self._get_chunks_path(chunk_size, chunk_overlap)
        print(f"Saving {len(chunks)} chunks to cache...")
        with open(path, 'wb') as f:
            pickle.dump(chunks, f, protocol=pickle.HIGHEST_PROTOCOL)
        print(f"Chunks saved: {os.path.basename(path)}")

    def load_chunks(self, chunk_size: int, chunk_overlap: int) -> List[Dict]:
        path = self._get_chunks_path(chunk_size, chunk_overlap)
        print(f"Loading chunks from cache...")
        with open(path, 'rb') as f:
            chunks = pickle.load(f)
        print(f" Loaded {len(chunks)} chunks from cache")
        return chunks

    def save_embeddings(
        self,
        embeddings: np.ndarray,
        chunk_metadata: Dict[int, Dict],
        chunk_size: int,
        chunk_overlap: int,
        model_name: str
    ):
        path = self._get_embeddings_path(chunk_size, chunk_overlap, model_name)
        print(f"Saving embeddings to cache ({embeddings.shape})...")
        data = {
            'embeddings': embeddings,
            'chunk_metadata': chunk_metadata,
            'model_name': model_name,
            'timestamp': datetime.now().isoformat()
        }
        with open(path, 'wb') as f:
            pickle.dump(data, f, protocol=pickle.HIGHEST_PROTOCOL)
        print(f" Embeddings saved: {os.path.basename(path)}")

    def load_embeddings(
        self,
        chunk_size: int,
        chunk_overlap: int,
        model_name: str
    ) -> Tuple[np.ndarray, Dict[int, Dict]]:
        path = self._get_embeddings_path(chunk_size, chunk_overlap, model_name)
        print(f"Loading embeddings from cache...")
        with open(path, 'rb') as f:
            data = pickle.load(f)
        embeddings = data['embeddings']
        chunk_metadata = data['chunk_metadata']
        print(f" Loaded embeddings from cache ({embeddings.shape})")
        return embeddings, chunk_metadata

print(" CacheManager class defined")

 CacheManager class defined


In [ ]:
class OptimizedRAGIndexer:
    def __init__(self, embedding_model_name: str, cache_manager: CacheManager):
        self.embedding_model_name = embedding_model_name
        self.cache_manager = cache_manager
        self.embedding_model = None
        self.cross_encoder = None
        self.index = None
        self.chunk_metadata = None
        self.chunk_embeddings = None
        self.is_built = False

    def load_embedding_model(self):
        if self.embedding_model is None:
            print(f"Loading embedding model: {self.embedding_model_name}")
            device = "cuda" if torch.cuda.is_available() else "cpu"
            self.embedding_model = SentenceTransformer(self.embedding_model_name, device=device)
            print(f" Model loaded on {device}")

    def load_cross_encoder(self):
        if self.cross_encoder is None:
            print("Loading cross-encoder...")
            device = "cuda" if torch.cuda.is_available() else "cpu"
            self.cross_encoder = CrossEncoder(
                "BAAI/bge-reranker-v2-m3",
                device=device,
                max_length=1024
            )
            print(f" Cross-encoder loaded")

    def build_index(
        self,
        documents: Dict[str, List[str]],
        doc_sources: Dict[str, List[PageSource]],
        chunk_size: int,
        chunk_overlap: int,
        force_rebuild: bool = False
    ) -> None:
        print(f"BUILDING INDEX (chunk_size={chunk_size}, overlap={chunk_overlap})")

        # Step 1: Get or create chunks
        if not force_rebuild and self.cache_manager.chunks_exist(chunk_size, chunk_overlap):
            chunks = self.cache_manager.load_chunks(chunk_size, chunk_overlap)
        else:
            print("Creating chunks from scratch...")
            chunks = split_documents(documents, doc_sources, chunk_size, chunk_overlap)
            self.cache_manager.save_chunks(chunks, chunk_size, chunk_overlap)

        # Step 2: Get or create embeddings
        if not force_rebuild and self.cache_manager.embeddings_exist(
            chunk_size, chunk_overlap, self.embedding_model_name
        ):
            embeddings, chunk_metadata = self.cache_manager.load_embeddings(
                chunk_size, chunk_overlap, self.embedding_model_name
            )
        else:
            print("Generating embeddings from scratch...")
            self.load_embedding_model()

            chunk_texts = [chunk["content"] for chunk in chunks]
            batch_size = 64 if torch.cuda.is_available() else 16

            embeddings = self.embedding_model.encode(
                chunk_texts,
                convert_to_numpy=True,
                show_progress_bar=True,
                batch_size=batch_size,
                normalize_embeddings=True
            ).astype("float32")

            print(f"Embeddings generated: {embeddings.shape}")

            chunk_metadata = {}
            for i, chunk in enumerate(chunks):
                chunk_metadata[i] = {
                    "source": chunk["source"],
                    "chunk_id": chunk["chunk_id"],
                    "content": chunk["content"],
                    "citation": chunk["citation"],
                    "word_count": chunk["word_count"],
                    "char_count": chunk["char_count"],
                    "page_number": chunk["page_number"]
                }

            self.cache_manager.save_embeddings(
                embeddings, chunk_metadata, chunk_size, chunk_overlap, self.embedding_model_name
            )

        # Step 3: Build FAISS index
        print("Building FAISS index...")
        dimension = embeddings.shape[1]
        index = faiss.IndexFlatIP(dimension)
        index.add(embeddings)
        print(f"FAISS index created: {index.ntotal} vectors")

        self.index = index
        self.chunk_metadata = chunk_metadata
        self.chunk_embeddings = embeddings
        self.is_built = True

        print(f" Index build complete")

    # ── FIX: load index from cache without needing documents ──────────────────
    def load_index_from_cache(self, chunk_size: int, chunk_overlap: int) -> bool:
        """Restore FAISS index from saved embeddings — no PDF documents required."""
        if not self.cache_manager.embeddings_exist(
            chunk_size, chunk_overlap, self.embedding_model_name
        ):
            print(" No cached embeddings found for these parameters.")
            return False

        embeddings, chunk_metadata = self.cache_manager.load_embeddings(
            chunk_size, chunk_overlap, self.embedding_model_name
        )

        print("Rebuilding FAISS index from cached embeddings...")
        dimension = embeddings.shape[1]
        index = faiss.IndexFlatIP(dimension)
        index.add(embeddings)

        self.index = index
        self.chunk_metadata = chunk_metadata
        self.chunk_embeddings = embeddings
        self.is_built = True

        print(f" Index loaded from cache ({index.ntotal} vectors)")
        return True
    # ─────────────────────────────────────────────────────────────────────────

    def get_index(self) -> Tuple[faiss.Index, Dict[int, Dict]]:
        if not self.is_built:
            raise ValueError("Index not built. Call build_index() or load_index_from_cache() first.")
        return self.index, self.chunk_metadata

    def rerank_results(
        self,
        query: str,
        initial_results: List[Dict],
        top_k: int = 5,
        batch_size: int = 32
    ) -> List[Dict]:
        if not initial_results:
            return []

        self.load_cross_encoder()

        pairs = [[query, chunk['content']] for chunk in initial_results]
        scores = self.cross_encoder.predict(pairs, batch_size=batch_size, show_progress_bar=False)

        for i, result in enumerate(initial_results):
            result['ce_score'] = float(scores[i])

        reranked = sorted(initial_results, key=lambda x: x['ce_score'], reverse=True)
        return reranked[:top_k]

print(" OptimizedRAGIndexer class defined")

 OptimizedRAGIndexer class defined


In [ ]:
class OptimizedParameterTuner:

    def __init__(self, indexer: OptimizedRAGIndexer, seed: int = 42):
        self.indexer = indexer
        self.results = []
        self.best_params = None
        self.best_score = 0.0
        random.seed(seed)
        torch.manual_seed(seed)

    PARAMETER_SPACE = {
        'chunk_size': [600, 800, 1000, 1200],
        'chunk_overlap': [100, 150, 200],
        'top_k': [5, 7, 10, 12],
        'rerank_enabled': [True, False],
    }

    def calculate_rougeL(self, prediction: str, reference: str) -> float:
        if not reference or not prediction:
            return 0.0
        scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)
        score = scorer.score(reference, prediction)
        return float(score['rougeL'].fmeasure)

    def tune_parameters(
        self,
        documents: Dict[str, List[str]],
        doc_sources: Dict[str, List[PageSource]],
        test_queries: List[str],
        ground_truths: List[str],
        model: Any,
        tokenizer: Any,
        num_combinations: int = 25,
        verbose: bool = True
    ) -> Optional[Dict]:

        print("OPTIMIZED HYPERPARAMETER TUNING (WITH CACHING)")
        print(f"Test queries: {len(test_queries)}")
        print(f"Metric: ROUGE-L F1")

        all_combinations = list(itertools.product(
            self.PARAMETER_SPACE['chunk_size'],
            self.PARAMETER_SPACE['chunk_overlap'],
            self.PARAMETER_SPACE['top_k'],
            self.PARAMETER_SPACE['rerank_enabled']
        ))

        if len(all_combinations) > num_combinations:
            test_combos = random.sample(all_combinations, num_combinations)
        else:
            test_combos = all_combinations

        print(f"Total possible combinations : {len(all_combinations)}")
        print(f"Testing {len(test_combos)} combinations...\n")

        combo_groups = {}
        for combo in test_combos:
            chunk_size, overlap, top_k, rerank = combo
            key = (chunk_size, overlap)
            if key not in combo_groups:
                combo_groups[key] = []
            combo_groups[key].append(combo)

        print(f" Optimized: Only {len(combo_groups)} index builds needed\n")

        for group_idx, ((chunk_size, overlap), group_combos) in enumerate(combo_groups.items(), 1):
            print(f"\n[Group {group_idx}/{len(combo_groups)}] Building index: size={chunk_size}, overlap={overlap}")

            self.indexer.build_index(
                documents=documents,
                doc_sources=doc_sources,
                chunk_size=chunk_size,
                chunk_overlap=overlap
            )

            for combo in tqdm(group_combos, desc=f"Group {group_idx} combos"):
                _, _, top_k, rerank = combo
                combo_scores = []

                for query, ground_truth in zip(test_queries, ground_truths):
                    try:
                        query_embedding = self.indexer.embedding_model.encode(
                            [query],
                            convert_to_numpy=True,
                            normalize_embeddings=True
                        ).astype("float32")

                        index, metadata = self.indexer.get_index()
                        retrieve_k = top_k * 2 if rerank else top_k
                        scores, indices = index.search(query_embedding, retrieve_k)

                        retrieved_chunks = []
                        for score, idx in zip(scores[0], indices[0]):
                            if idx == -1 or idx >= len(metadata):
                                continue
                            meta = metadata.get(idx, {})
                            retrieved_chunks.append({
                                "content": meta.get("content", ""),
                                "faiss_similarity": float(score)
                            })

                        if rerank and len(retrieved_chunks) > top_k:
                            self.indexer.load_cross_encoder()
                            pairs = [[query, c['content']] for c in retrieved_chunks]
                            rerank_scores = self.indexer.cross_encoder.predict(pairs, show_progress_bar=False)
                            for i, chunk in enumerate(retrieved_chunks):
                                chunk['ce_score'] = float(rerank_scores[i])
                            retrieved_chunks = sorted(
                                retrieved_chunks,
                                key=lambda x: x['ce_score'],
                                reverse=True
                            )[:top_k]

                        context = "\n\n".join([c['content'] for c in retrieved_chunks])

                        prompt = f"""You are an expert orthopedic surgeon. Answer the question directly and concisely using only the provided context.

CONTEXT:
{context}

STRICT RULES:
1. Answer DIRECTLY — do NOT show your reasoning, analysis steps, or thought process.
2. Do NOT use headings like "Analyze", "Synthesize", "Review Sources", or "Final Check".
3. Use numbered bullet points only for the actual answer steps.
4. If unsure, state uncertainty explicitly.
5. Do NOT fabricate specific statistics, references, or guidelines.
6. Do NOT speculate beyond standard clinical practice.
7. Maximum 300 words.

QUESTION:
{query}

ANSWER (direct, no reasoning steps):
"""
                        inputs = tokenizer(
                            prompt,
                            return_tensors="pt",
                            truncation=True,
                            max_length=8192
                        ).to(model.device)

                        with torch.no_grad():
                            outputs = model.generate(
                                **inputs,
                                max_new_tokens=2000,
                                temperature=0.0,
                                do_sample=False,
                                pad_token_id=tokenizer.eos_token_id
                            )

                        generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]
                        answer = tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()

                        think_match = re.search(r'</think>(.*)', answer, re.DOTALL)
                        if think_match:
                            answer = think_match.group(1).strip()

                        rougeL_score = self.calculate_rougeL(answer, ground_truth)
                        combo_scores.append(rougeL_score)

                        if verbose:
                            print(f"   ROUGE-L F1: {rougeL_score:.3f} | Query: {query[:60]}...")

                    except Exception as e:
                        print(f"\nError: {e}")
                        combo_scores.append(0.0)

                avg_score = sum(combo_scores) / len(combo_scores) if combo_scores else 0.0

                result_entry = {
                    "params": {
                        "chunk_size":     chunk_size,
                        "chunk_overlap":  overlap,
                        "top_k":          top_k,
                        "rerank_enabled": rerank,
                    },
                    "avg_rougeL_f1": avg_score,
                    "query_scores":  combo_scores,
                    "status":        "success"
                }
                self.results.append(result_entry)

                if avg_score > self.best_score:
                    self.best_score = avg_score
                    self.best_params = result_entry["params"]
                    if verbose:
                        print(f"\n New best! ROUGE-L F1={avg_score:.3f}, params={self.best_params}")

            if torch.cuda.is_available():
                torch.cuda.empty_cache()

        print("TUNING COMPLETE")
        print(f"Best ROUGE-L F1 : {self.best_score:.3f}")
        print(f"Best Params     : {self.best_params}")

        return self.best_params

print(" OptimizedParameterTuner class defined")

 OptimizedParameterTuner class defined


In [ ]:
class ComprehensiveEvaluator:
    def __init__(self, experiment_logger: ExperimentLogger):
        self.logger = experiment_logger
        self.rouge_scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
        self.meteor_metric = evaluate.load("meteor")

    def calculate_rouge(self, prediction: str, reference: str) -> Dict:
        if not reference or not reference.strip():
            return {
                'rouge1': {'precision': 0, 'recall': 0, 'fmeasure': 0},
                'rouge2': {'precision': 0, 'recall': 0, 'fmeasure': 0},
                'rougeL': {'precision': 0, 'recall': 0, 'fmeasure': 0}
            }
        scores = self.rouge_scorer.score(reference, prediction)
        return {
            'rouge1': {'precision': scores['rouge1'].precision, 'recall': scores['rouge1'].recall, 'fmeasure': scores['rouge1'].fmeasure},
            'rouge2': {'precision': scores['rouge2'].precision, 'recall': scores['rouge2'].recall, 'fmeasure': scores['rouge2'].fmeasure},
            'rougeL': {'precision': scores['rougeL'].precision, 'recall': scores['rougeL'].recall, 'fmeasure': scores['rougeL'].fmeasure}
        }

    def calculate_bertscore(self, prediction: str, reference: str) -> Dict[str, float]:
        if not reference or not prediction:
            return {'precision': 0.0, 'recall': 0.0, 'f1': 0.0}
        try:
            P, R, F1 = bert_score([prediction], [reference], lang="en", verbose=False, device="cpu")
            return {'precision': float(P[0]), 'recall': float(R[0]), 'f1': float(F1[0])}
        except:
            return {'precision': 0.0, 'recall': 0.0, 'f1': 0.0}

    def calculate_meteor(self, prediction: str, reference: str) -> float:
        if not reference:
            return 0.0
        try:
            result = self.meteor_metric.compute(predictions=[prediction], references=[reference])
            return float(result['meteor'])
        except:
            return 0.0

    def run_ablation_study(
        self,
        indexer: OptimizedRAGIndexer,
        model: Any,
        tokenizer: Any,
        queries: List[str],
        ground_truths: Optional[List[str]] = None,
        hyperparameters: Optional[Dict[str, Any]] = None
    ) -> Dict[str, Any]:

        if hyperparameters is None:
            hyperparameters = {
                'chunk_size': 800, 'chunk_overlap': 150, 'top_k': 7,
                'rerank_enabled': True, 'temperature': 0.0,
                'embedding_model': EMBEDDING_MODEL_NAME, 'llm_model': LLM_MODEL_NAME
            }

        if ground_truths is None:
            ground_truths = [None] * len(queries)

        print("=" * 70)
        print("ABLATION STUDY: RAG vs Vanilla")
        print("=" * 70)
        print(f"Queries           : {len(queries)}")
        print(f"chunk_size        : {hyperparameters.get('chunk_size')}")
        print(f"chunk_overlap     : {hyperparameters.get('chunk_overlap')}")
        print(f"top_k             : {hyperparameters.get('top_k')}")
        print(f"rerank_enabled    : {hyperparameters.get('rerank_enabled')}")
        print(f"temperature       : {hyperparameters.get('temperature')}")
        print("=" * 70 + "\n")

        configurations = [
            {'name': 'RAG',     'use_rag': True},
            {'name': 'Vanilla', 'use_rag': False}
        ]

        all_results = []

        for config in configurations:
            config_name = config['name']
            print(f"\n{'='*70}")
            print(f"CONFIGURATION: {config_name}")
            print(f"{'='*70}\n")

            for idx, (query, ground_truth) in enumerate(zip(queries, ground_truths), 1):
                print(f"Query {idx}/{len(queries)}: {query[:80]}...")

                try:
                    if config['use_rag']:
                        result = rag_generate(
                            indexer, model, tokenizer, query,
                            k=hyperparameters['top_k'],
                            use_reranking=hyperparameters['rerank_enabled'],
                            verbose=False
                        )
                    else:
                        result = vanilla_query(model, tokenizer, query, verbose=False)
                        result['retrieval_metrics'] = {}
                        result['citations'] = []

                    rouge_scores      = None
                    bertscore_metrics = None
                    meteor_score      = None

                    answer = result.get('answer') or ''

                    # Safety strip: catch any residual thinking block not cleaned upstream
                    think_match = re.search(r'</think>(.*)', answer, re.DOTALL)
                    if think_match:
                        answer = think_match.group(1).strip()
                        result['answer'] = answer

                    # Warn if answer is suspiciously short after stripping
                    if len(answer.split()) < 10:
                        print(f"   WARNING: Very short answer ({len(answer.split())} words) — possible generation failure")

                    print(f"   Answer preview : {answer[:120].strip()}...")

                    if ground_truth and ground_truth.strip() and answer:
                        rouge_scores      = self.calculate_rouge(answer, ground_truth)
                        bertscore_metrics = self.calculate_bertscore(answer, ground_truth)
                        meteor_score      = self.calculate_meteor(answer, ground_truth)
                        print(f"   ROUGE-1 F1     : {rouge_scores['rouge1']['fmeasure']:.3f} | "
                              f"BERTScore F1: {bertscore_metrics['f1']:.3f} | "
                              f"METEOR: {meteor_score:.3f}")

                    self.logger.log_experiment(
                        query=query,
                        answer=answer,
                        configuration=config_name,
                        ground_truth=ground_truth,
                        rouge_scores=rouge_scores,
                        bertscore_metrics=bertscore_metrics,
                        meteor_score=meteor_score,
                        retrieval_metrics=result.get('retrieval_metrics', {}),
                        citations=result.get('citations', []),
                        hyperparameters=hyperparameters,
                        execution_time=result.get('execution_time', 0),
                        status=result.get('status', 'unknown'),
                        error_message=result.get('error', ''),
                        experiment_id=f"{config_name}_{idx:03d}"
                    )

                    all_results.append({
                        'configuration': config_name,
                        'query_idx': idx,
                        'result': result
                    })

                    print(f"   Status         : {result.get('status')}")

                except Exception as e:
                    print(f"   Error: {e}")

        print("ABLATION STUDY COMPLETE")

        return {
            'all_results': all_results,
            'configurations': [c['name'] for c in configurations],
            'num_queries': len(queries)
        }

print(" ComprehensiveEvaluator class defined")

 ComprehensiveEvaluator class defined


In [ ]:
def clean_text(text: str, lowercase: bool = False, preserve_newlines: bool = False) -> str:
    if not text or not isinstance(text, str):
        return ""
    # Remove only true control characters; preserve all printable Unicode (incl. Greek, superscripts)
    text = re.sub(r'[\x00-\x08\x0B\x0C\x0E-\x1F\x7F]', ' ', text)
    if preserve_newlines:
        lines = text.split('\n')
        cleaned_lines = []
        for line in lines:
            line = re.sub(r'[ \t]+', ' ', line).strip()
            if line:
                cleaned_lines.append(line)
        text = "\n".join(cleaned_lines)
    else:
        text = re.sub(r'\s+', ' ', text).strip()

    text = re.sub(
        r'[^\w\s\.\,\;\:\-\+\(\)\/\%\=\<\>\°\±\[\]\'\"\!\?\@\#\&\*\~\`\^\{\}\|\\]',
        '',
        text,
        flags=re.UNICODE
    )
    if lowercase:
        text = text.lower()
    return text


def read_pdf(
    file_path: str,
    book_title: str,
    author: Optional[str] = None,
    publication_year: Optional[int] = None
) -> Tuple[List[str], List[PageSource]]:
    """
    Extract text from a medical PDF using pdfplumber, which handles multi-column
    layouts, tables, and footnotes far more faithfully than PyPDF2.
    """
    if not os.path.exists(file_path):
        return [], []
    try:
        texts: List[str] = []
        sources: List[PageSource] = []
        with pdfplumber.open(file_path) as pdf:
            for page_num, page in enumerate(pdf.pages):
                try:
                    # Extract plain text; pdfplumber respects reading order and columns
                    raw_text = page.extract_text(x_tolerance=2, y_tolerance=3)
                    if raw_text is None or len(raw_text.strip()) < 100:
                        continue
                    cleaned_text = clean_text(raw_text, lowercase=False, preserve_newlines=False)
                    if not cleaned_text:
                        continue
                    source = PageSource(
                        document_name=os.path.basename(file_path),
                        page_number=page_num + 1,
                        book_title=book_title,
                        author=author,
                        publication_year=publication_year
                    )
                    texts.append(cleaned_text)
                    sources.append(source)
                except Exception:
                    continue
        return texts, sources
    except Exception as e:
        print(f"Error reading PDF {file_path}: {e}")
        return [], []


def load_documents(
    folder_path: str,
    doc_metadata: Dict[str, dict]
) -> Tuple[Dict[str, List[str]], Dict[str, List[PageSource]]]:
    documents: Dict[str, List[str]] = {}
    doc_sources: Dict[str, List[PageSource]] = {}

    if not os.path.isdir(folder_path):
        return documents, doc_sources

    files = [f for f in os.listdir(folder_path) if f.lower().endswith(('.pdf', '.txt', '.md'))]
    print(f"Loading {len(files)} documents...")

    for filename in tqdm(files, desc="Loading"):
        file_path = os.path.join(folder_path, filename)
        metadata = doc_metadata.get(filename, {})
        try:
            if filename.lower().endswith('.pdf'):
                texts, sources = read_pdf(
                    file_path=file_path,
                    book_title=metadata.get('book_title', filename),
                    author=metadata.get('author'),
                    publication_year=metadata.get('year')
                )
                if texts:
                    documents[filename] = texts
                    doc_sources[filename] = sources
        except Exception as e:
            print(f"Skipping {filename}: {e}")

    print(f" Loaded {len(documents)} documents\n")
    return documents, doc_sources


def split_documents(
    documents: Dict[str, List[str]],
    doc_sources: Dict[str, List[PageSource]],
    chunk_size: int = 800,
    chunk_overlap: int = 150
) -> List[Dict]:
    if not documents:
        return []
    if chunk_overlap >= chunk_size:
        chunk_overlap = chunk_size // 5

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", ". ", "; ", ", ", " ", ""],
        length_function=len,
        keep_separator=True  # Avoids cutting mid-sentence at separator boundaries
    )

    all_chunks = []
    for doc_name in tqdm(documents.keys(), desc="Chunking"):
        page_texts = documents[doc_name]
        sources = doc_sources.get(doc_name, [])
        if not sources:
            continue
        for page_idx, page_text in enumerate(page_texts):
            if page_idx >= len(sources):
                continue
            page_source = sources[page_idx]
            # text was already cleaned by clean_text() inside read_pdf(); no second pass needed
            if not page_text or len(page_text) < 50:
                continue
            chunks = text_splitter.split_text(page_text)
            for chunk_id, chunk_text in enumerate(chunks):
                chunk_obj = {
                    'source': doc_name,
                    'page_number': page_source.page_number,
                    'chunk_id': f"{doc_name}_p{page_source.page_number}_c{chunk_id}",
                    'content': chunk_text,
                    'char_count': len(chunk_text),
                    'word_count': len(chunk_text.split()),
                    'citation': page_source.get_citation(),
                }
                all_chunks.append(chunk_obj)

    print(f" Created {len(all_chunks)} chunks")
    return all_chunks

In [ ]:
def rag_query(
    indexer: OptimizedRAGIndexer,
    query: str,
    k: int = 5,
    use_reranking: bool = True,
    verbose: bool = False
) -> Dict[str, Any]:
    try:
        index, metadata = indexer.get_index()

        query_embedding = indexer.embedding_model.encode(
            [query],
            convert_to_numpy=True,
            normalize_embeddings=True
        ).astype("float32")

        retrieve_k = k * 2 if use_reranking else k
        scores, indices = index.search(query_embedding, retrieve_k)

        retrieved_chunks = []
        for score, idx in zip(scores[0], indices[0]):
            if idx == -1 or idx >= len(metadata):
                continue
            meta = metadata.get(idx, {})
            retrieved_chunks.append({
                "index": int(idx),
                "source": meta.get("source", "Unknown"),
                "content": meta.get("content", ""),
                "citation": meta.get("citation", "Unknown"),
                "faiss_similarity": float(score),
                "page_number": meta.get("page_number", 0)
            })

        if use_reranking and len(retrieved_chunks) > k:
            retrieved_chunks = indexer.rerank_results(query, retrieved_chunks, top_k=k)

        faiss_similarities = [c['faiss_similarity'] for c in retrieved_chunks]

        retrieval_metrics = {
            "total_chunks": len(retrieved_chunks),
            "reranking_applied": use_reranking,
            "faiss_top_similarity": max(faiss_similarities) if faiss_similarities else 0,
            "faiss_avg_similarity": sum(faiss_similarities) / len(faiss_similarities) if faiss_similarities else 0,
            "faiss_min_similarity": min(faiss_similarities) if faiss_similarities else 0,
            "faiss_max_similarity": max(faiss_similarities) if faiss_similarities else 0,
        }

        if use_reranking:
            ce_scores = [c.get('ce_score', 0) for c in retrieved_chunks]
            retrieval_metrics.update({
                "ce_top_score": max(ce_scores) if ce_scores else 0,
                "ce_avg_score": sum(ce_scores) / len(ce_scores) if ce_scores else 0,
                "ce_min_score": min(ce_scores) if ce_scores else 0,
                "ce_max_score": max(ce_scores) if ce_scores else 0,
            })
        else:
            retrieval_metrics.update({
                "ce_top_score": 0, "ce_avg_score": 0,
                "ce_min_score": 0, "ce_max_score": 0,
            })

        citations = list(dict.fromkeys([
            chunk.get("citation")
            for chunk in retrieved_chunks
            if chunk.get("citation")
        ]))

        context_chunks = [
            f"[Source: {chunk['citation']}]\n{chunk['content']}"
            for chunk in retrieved_chunks
        ]
        context = "\n\n".join(context_chunks)

        return {
            "query": query,
            "context": context,
            "citations": citations,
            "retrieval_metrics": retrieval_metrics,
            "status": "ready_for_generation",
            "retrieved_chunks": retrieved_chunks
        }

    except Exception as e:
        return {
            "query": query,
            "answer": None,
            "retrieval_metrics": {},
            "status": "error",
            "error": str(e),
        }

In [ ]:
def rag_generate(
    indexer: OptimizedRAGIndexer,
    model: Any,
    tokenizer: Any,
    query: str,
    k: int = 5,
    use_reranking: bool = True,
    verbose: bool = False
) -> Dict[str, Any]:
    start_time = datetime.now()
    try:
        rag_result = rag_query(indexer, query, k, use_reranking, verbose)

        if rag_result["status"] == "error":
            return rag_result

        context = rag_result["context"]

        prompt = f"""You are an expert orthopedic surgeon. Answer the question directly and concisely using only the provided context.

CONTEXT:
{context}

STRICT RULES:
1. Answer DIRECTLY — do NOT show your reasoning, analysis steps, or thought process.
2. Do NOT use headings like "Analyze", "Synthesize", "Review Sources", or "Final Check".
3. Use numbered bullet points only for the actual answer steps.
4. If unsure, state uncertainty explicitly.
5. Do NOT fabricate specific statistics, references, or guidelines.
6. Do NOT speculate beyond standard clinical practice.
7. Maximum 300 words.


QUESTION:
{query}

ANSWER (direct, no reasoning steps):
"""
        inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=8192).to(model.device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=2000,
                temperature=0.0,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id
            )

        generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]
        answer = tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()

        execution_time = (datetime.now() - start_time).total_seconds()

        return {
            "query": query,
            "answer": answer,
            "context": context,
            "citations": rag_result["citations"],
            "retrieval_metrics": rag_result["retrieval_metrics"],
            "status": "success",
            "execution_time": execution_time,
            "retrieved_chunks": rag_result.get("retrieved_chunks", [])
        }

    except Exception as e:
        execution_time = (datetime.now() - start_time).total_seconds()
        return {
            "query": query,
            "answer": None,
            "citations": [],
            "retrieval_metrics": {},
            "status": "error",
            "error": str(e),
            "execution_time": execution_time
        }

In [ ]:
def vanilla_query(
    model: Any,
    tokenizer: Any,
    query: str,
    verbose: bool = False
) -> Dict[str, Any]:
    start_time = datetime.now()

    prompt = f"""You are an expert orthopedic surgeon. Answer the question directly and concisely based on established orthopedic medical knowledge.

STRICT RULES:
1. Answer DIRECTLY — do NOT show your reasoning, analysis steps, or thought process.
2. Do NOT use headings like "Analyze", "Synthesize", "Review Sources", or "Final Check".
3. Use numbered bullet points only for the actual answer steps.
4. If unsure, state uncertainty explicitly.
5. Do NOT fabricate specific statistics, references, or guidelines.
6. Do NOT speculate beyond standard clinical practice.
7. Maximum 300 words.

QUESTION:
{query}

ANSWER (direct, no reasoning steps):
"""
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=8192).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=2000,
            temperature=0.0,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    answer = tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()

    execution_time = (datetime.now() - start_time).total_seconds()

    return {
        "query": query,
        "answer": answer,
        "model": "vanilla_medgemma",
        "execution_time": execution_time,
        "status": "success"
    }

print(" All helper functions defined\n")

 All helper functions defined



In [ ]:
print("MOUNTING GOOGLE DRIVE")

if not os.path.ismount("/content/drive"):
    drive.mount("/content/drive")
    print(" Drive mounted\n")
else:
    print(" Drive already mounted\n")

MOUNTING GOOGLE DRIVE
Mounted at /content/drive
 Drive mounted



In [ ]:
os.makedirs(DOC_FOLDER, exist_ok=True)
os.makedirs(CACHE_FOLDER, exist_ok=True)
os.makedirs(os.path.dirname(EXPERIMENT_LOG_PATH), exist_ok=True)

DATA_FOLDER_ID = "1C8_jMFaiIz6JS-YzeAuvqXu-600sQSGE"

doc_files = [f for f in os.listdir(DOC_FOLDER) if f.lower().endswith('.pdf')] if os.path.exists(DOC_FOLDER) else []

if len(doc_files) < 8:
    print("Downloading documents from Google Drive...\n")
    try:
        auth.authenticate_user()
        drive_service = build("drive", "v3")

        response = drive_service.files().list(
            q=f"'{DATA_FOLDER_ID}' in parents and trashed=false",
            spaces="drive",
            fields="files(id, name)",
            pageSize=100
        ).execute()

        drive_files = response.get("files", [])

        for file in tqdm(drive_files, desc="Downloading"):
            file_path = os.path.join(DOC_FOLDER, file["name"])
            if os.path.exists(file_path):
                continue
            request = drive_service.files().get_media(fileId=file["id"])
            file_stream = io.BytesIO()
            downloader = MediaIoBaseDownload(file_stream, request)
            done = False
            while not done:
                status, done = downloader.next_chunk()
            with open(file_path, "wb") as f:
                f.write(file_stream.getvalue())

        print(" Documents downloaded\n")

    except Exception as e:
        print(f" Error downloading: {e}")
        print("Please manually upload PDFs to", DOC_FOLDER, "\n")
else:
    print(f" Documents already present ({len(doc_files)} files)\n")

 Documents already present (8 files)



In [ ]:
print("INITIALIZING COMPONENTS")
cache_manager     = CacheManager(CACHE_FOLDER)
print(" Cache manager ready")
# Fix 2a: clear_existing=True — fresh log every run
experiment_logger = ExperimentLogger(EXPERIMENT_LOG_PATH, clear_existing=True)
print(" Experiment logger ready")

INITIALIZING COMPONENTS
 Cache manager ready
 Existing log file 'final_experiments_optimized.xlsx' cleared.
 Experiment logger ready


In [ ]:
print("Loading documents (skip this block if embeddings cache already exists)...")
documents, doc_sources = load_documents(DOC_FOLDER, doc_metadata)
if not documents:
    print(" WARNING: No documents loaded — Block 33 will use cache fallback")
else:
    print(f" {len(documents)} documents loaded")

Loading documents (skip this block if embeddings cache already exists)...
Loading 8 documents...


Loading: 100%|██████████| 8/8 [21:18<00:00, 159.80s/it]

 Loaded 8 documents

 8 documents loaded


In [ ]:
indexer = OptimizedRAGIndexer(EMBEDDING_MODEL_NAME, cache_manager)
print(" Indexer object created")

 Indexer object created


In [ ]:
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(f"  GPU   : {torch.cuda.get_device_name(0)}")
    print(f"  VRAM  : {torch.cuda.get_device_properties(0).total_memory/(1024**3):.1f} GB")
torch.set_grad_enabled(False)

  GPU   : NVIDIA RTX PRO 6000 Blackwell Server Edition
  VRAM  : 95.0 GB


torch.autograd.grad_mode.set_grad_enabled(mode=False)

In [ ]:
from huggingface_hub import login
login()

In [ ]:
print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL_NAME, trust_remote_code=True)
print(" Tokenizer loaded\n")

Loading tokenizer...


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

 Tokenizer loaded



In [ ]:
print("Loading LLM...")
model = AutoModelForCausalLM.from_pretrained(
    LLM_MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
    attn_implementation="eager",
)
model.eval()
print(" Model loaded\n")

Loading LLM...


`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 12 files:   0%|          | 0/12 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/1247 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

 Model loaded



In [ ]:
TUNING_QUERIES = [
    # Clinical vignette requiring intraoperative reasoning
    "During a total knee replacement in a 72-year-old woman with severe varus deformity, after bone cuts the flexion gap is balanced but the extension gap is tight medially. The knee cannot be fully extended. What sequential steps should be taken to achieve a balanced extension gap?",

    # Multi-part diagnostic + management question
    "A 70-year-old patient presents 6 weeks after total knee replacement with a fever of 38.6°C, persistent wound drainage, and elevated CRP of 180 mg/L. What criteria would you use to confirm periprosthetic joint infection, and what is your management algorithm?",

    # Implant selection + biomechanics reasoning
    "A 58-year-old active male with medial compartment osteoarthritis, intact ACL, correctable 8-degree varus deformity, and BMI of 28 is being considered for knee arthroplasty. Compare the outcomes of unicompartmental knee arthroplasty versus total knee arthroplasty for this patient and justify your choice.",

    # Complication-focused clinical scenario
    "A 65-year-old patient develops sudden onset of severe pain and inability to extend the knee 3 weeks after total knee arthroplasty. Examination shows a palpable defect above the patella. What is the diagnosis and describe your management?",

    # Technical decision-making question
    "In a patient undergoing revision total knee arthroplasty, intraoperative assessment reveals an Anderson Orthopaedic Research Institute Type 2B tibial defect. What reconstruction options are available, and what factors determine your choice between a structural allograft and a trabecular metal cone?",
]

TUNING_GROUND_TRUTHS = [
    """When the extension gap is tight medially after balanced flexion gap in a varus TKR, sequential releases are performed in a stepwise manner: first, osteophytes from the posteromedial tibia and femur are excised as they are a common occult cause of extension tightness; second, the superficial MCL is progressively released from its tibial attachment using pie-crusting or a formal release at the posteromedial corner; third, the posteromedial capsule is released; fourth, if extension remains deficient, a further distal femoral resection of 2mm can be performed, which selectively opens the extension gap without altering the flexion gap; fifth, if instability in flexion results from the added distal femoral cut, a thicker polyethylene insert can compensate. Throughout these steps, the flexion gap is reassessed to ensure symmetry, and a constrained implant is used only if satisfactory balance cannot be achieved with soft tissue techniques alone.""",

    """Periprosthetic joint infection (PJI) is confirmed using the 2018 ICM criteria: major criteria (two positive cultures with the same organism, or a sinus tract communicating with the joint) are diagnostic alone; minor criteria include elevated serum CRP or D-dimer, elevated ESR, elevated synovial WBC (>3000 cells/uL) or positive leukocyte esterase, elevated synovial PMN% (>80%), positive histology (>5 neutrophils per high-power field), and a single positive culture. A score >= 6 confirms infection. Management depends on timing and organism: within 90 days of primary implant or <3 weeks of symptom onset with a stable well-fixed implant, DAIR (debridement, antibiotics, and implant retention) with exchange of modular components is appropriate; for chronic or late infection with a loose component, two-stage revision (explant + antibiotic spacer, followed by reimplantation after 6-12 weeks of culture-directed IV antibiotics) is the gold standard; suppressive antibiotics or one-stage revision may be appropriate in selected cases.""",

    """For a 58-year-old active male with isolated medial OA, intact ACL, correctable varus <10 degrees, and BMI <30, unicompartmental knee arthroplasty (UKA) is a strong candidate given that all classic Oxford criteria are met. UKA advantages in this setting include preservation of both cruciate ligaments, more natural kinematics, faster recovery, lower perioperative morbidity, and preserved bone stock facilitating easier future revision. However, TKA provides a more predictable and durable outcome in higher-demand patients, with lower revision rates at 15-20 years in population registries. Long-term survival of UKA at 10 years is approximately 90-95% in appropriately selected patients. Given his age, activity level, and ideal selection criteria, UKA is preferred as the index procedure, with TKA retained as a reliable revision option; patient preference and surgeon volume/experience with UKA should also inform the final decision.""",

    """The clinical picture - acute inability to extend the knee with a palpable suprapatellar gap 3 weeks post-TKA - is diagnostic of quadriceps tendon rupture. This is an uncommon but serious extensor mechanism complication with a reported incidence of 0.1-1.1% after TKA, associated with systemic steroids, rheumatoid arthritis, and intraoperative devascularization. Management: urgent MRI confirms diagnosis and assesses extent of tear. Primary repair is performed acutely with heavy non-absorbable sutures (e.g., FiberWire) in Krackow configuration with transosseous fixation through the patella; augmentation using a hamstring autograft, synthetic mesh, or allograft is added when tissue quality is poor. Postoperatively, the knee is immobilized in extension for 6 weeks, followed by a graduated physiotherapy protocol. Late or neglected ruptures may require allograft reconstruction of the entire extensor mechanism.""",

    """An AORI Type 2B tibial defect involves metaphyseal bone loss affecting both cortices of one compartment with compromise of the metaphyseal flare. Reconstruction options include: (1) Modular metal augments (wedges/blocks) for contained defects up to 15-20 mm - simple, reproducible, and easily revised; (2) Structural allografts (femoral head or bulk tibia) for larger uncontained defects - restores bone stock but carries risk of resorption, fracture, and infection; (3) Trabecular metal (tantalum) cones or tibial sleeves - preferred in most modern revision practices for type 2B defects as they achieve reliable biological fixation through bony ingrowth, provide rotational stability, and are more forgiving of cavitary defects; (4) Cemented stems without augmentation for very contained small defects. Choice between allograft and tantalum cone depends on defect size, containment, patient age (bone stock restoration preferred in younger patients), infection risk, and surgeon familiarity; tantalum cones are currently preferred in most centres due to superior fixation and lower complication profile compared to structural allografts.""",
]
print(f" Tuning queries defined ({len(TUNING_QUERIES)} queries)")

 Tuning queries defined (5 queries)


In [ ]:
print("HYPERPARAMETER TUNING CONFIGURATION")
print(f"  Test queries   : {len(TUNING_QUERIES)}")
print(f"  Combinations   : 96 (or all available)")
print(f"  Metric         : ROUGE-L F1")
print(f"  Cache          : ENABLED")
print()

HYPERPARAMETER TUNING CONFIGURATION
  Test queries   : 5
  Combinations   : 96 (or all available)
  Metric         : ROUGE-L F1
  Cache          : ENABLED



In [ ]:
ENABLE_TUNING           = False
TUNING_RAN_THIS_SESSION = False

if ENABLE_TUNING:
    if not documents:
        print(" WARNING: No documents loaded — cannot run tuning")
        print(" TIP: Run Block 23 first if you need to re-tune\n")
    else:
        print("Starting hyperparameter tuning...\n")
        tuner      = OptimizedParameterTuner(indexer, seed=RANDOM_SEED)
        best_params = tuner.tune_parameters(
            documents=documents, doc_sources=doc_sources,
            test_queries=TUNING_QUERIES, ground_truths=TUNING_GROUND_TRUTHS,
            model=model, tokenizer=tokenizer,
            num_combinations=25, verbose=True)

        if best_params:
            best_params['temperature']     = 0.0
            best_params['embedding_model'] = EMBEDDING_MODEL_NAME
            best_params['llm_model']       = LLM_MODEL_NAME
            with open(TUNED_PARAMS_PATH, "w") as f:
                json.dump({
                    "best_parameters":         best_params,
                    "quality_score":           float(tuner.best_score),
                    "metric":                  "rougeL_f1",
                    "timestamp":               datetime.now().isoformat(),
                    "num_combinations_tested": len(tuner.results),
                    "num_tuning_queries":      len(TUNING_QUERIES),
                    "all_results":             tuner.results,
                }, f, indent=2)
            print(f"\n Best parameters saved to: {TUNED_PARAMS_PATH}")
            print(f"  ROUGE-L F1   : {tuner.best_score:.4f}")
            print(f"  Best params  : {best_params}\n")
            BEST_PARAMS             = best_params
            TUNING_RAN_THIS_SESSION = True
        else:
            print(" Tuning failed — will use default / previously saved parameters")
else:
    print("Tuning DISABLED (ENABLE_TUNING=False) — using saved or default params\n")

Tuning DISABLED (ENABLE_TUNING=False) — using saved or default params



In [ ]:
print("LOADING BEST PARAMETERS")

if not TUNING_RAN_THIS_SESSION:
    if os.path.exists(TUNED_PARAMS_PATH):
        try:
            with open(TUNED_PARAMS_PATH, 'r') as f:
                pd_data = json.load(f)
            BEST_PARAMS = pd_data['best_parameters']
            print(f" Loaded from saved file")
            print(f"  {pd_data.get('metric','score')}: {pd_data.get('quality_score',0):.4f}")
        except Exception as e:
            print(f" Could not load saved params ({e}) — using defaults")
            BEST_PARAMS = {'chunk_size':1200,'chunk_overlap':150,'top_k':12,
                           'rerank_enabled':True,'temperature':0.0,
                           'embedding_model':EMBEDDING_MODEL_NAME,'llm_model':LLM_MODEL_NAME}
    else:
        print(" No saved params found — using defaults")
        BEST_PARAMS = {'chunk_size':1200,'chunk_overlap':150,'top_k':12,
                       'rerank_enabled':True,'temperature':0.0,
                       'embedding_model':EMBEDDING_MODEL_NAME,'llm_model':LLM_MODEL_NAME}
else:
    print(" Using parameters from tuning that just ran this session")

print("\nFinal Parameters:")
for k, v in BEST_PARAMS.items():
    print(f"  {k}: {v}")
print()

LOADING BEST PARAMETERS
 Loaded from saved file
  rougeL_f1: 0.2293

Final Parameters:
  chunk_size: 1000
  chunk_overlap: 100
  top_k: 10
  rerank_enabled: False
  temperature: 0.0
  embedding_model: BAAI/bge-m3
  llm_model: google/medgemma-27b-it



In [ ]:
print("BUILDING INDEX")

# Guard: if Block 23 was skipped, these variables may not exist
if 'documents'   not in dir(): documents   = {}
if 'doc_sources' not in dir(): doc_sources = {}

cs = BEST_PARAMS['chunk_size']
co = BEST_PARAMS['chunk_overlap']

if documents:
    # Path A: documents available → build (uses cache internally if already saved)
    print(" Path A: documents loaded — building/loading index")
    indexer.build_index(
        documents=documents, doc_sources=doc_sources,
        chunk_size=cs, chunk_overlap=co)
    indexer.load_embedding_model()
    print(" Index ready (Path A)\n")

elif cache_manager.embeddings_exist(cs, co, EMBEDDING_MODEL_NAME):
    # Path B: no documents but cache file exists → load directly
    print(" Path B: no documents in memory, but embeddings cache found")
    print(f"  Loading index for chunk_size={cs}, overlap={co}...")
    loaded = indexer.load_index_from_cache(chunk_size=cs, chunk_overlap=co)
    if loaded:
        indexer.load_embedding_model()
        print(" Index restored from cache (Path B)\n")
    else:
        print(" ERROR: cache load failed unexpectedly")

else:
    # Path C: nothing available → user must run Block 23
    print(" ERROR (Path C): No documents and no matching embeddings cache found.")
    print(f"  Expected cache for chunk_size={cs}, overlap={co}")
    print(f"  Cache folder : {CACHE_FOLDER}")
    print(f"  Solution     : Run Block 23 to load the PDF documents,")
    print(f"                 then re-run this block.")

print(f"  Indexer ready: {indexer.is_built}")
print(f"  GPU: {torch.cuda.get_device_name(0)}")
print(f"  VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

BUILDING INDEX
 Path A: documents loaded — building/loading index
BUILDING INDEX (chunk_size=1000, overlap=100)
 Found cached chunks: chunks_size1000_overlap100.pkl
Loading chunks from cache...
 Loaded 53568 chunks from cache
 Found cached embeddings: embeddings_size1000_overlap100_bge-m3.pkl
Loading embeddings from cache...
 Loaded embeddings from cache ((53568, 1024))
Building FAISS index...
FAISS index created: 53568 vectors
 Index build complete
 Index ready (Path A)

  Indexer ready: True
  GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition
  VRAM: 102.0 GB


In [ ]:
print("RUNNING EVALUATION")

TEST_QUERIES = [
    "After insertion of the trial components in a total knee replacement, the surgeon finds that he is unable to fully extend the knee and that the tibial tray lifts-off when the knee is flexed past 90 degrees. What intervention should be taken to achieve a knee that is balanced in flexion and extension?",
    "A 40-year-old man complains of increasing groin pain. Radiographs show femoral head avascular necrosis with subchondral lucency but without femoral head collapse. Which of the following medical treatments have been shown to decrease the risk of subsequent femoral head collapse?",
    "Enumerate the important operative steps in total knee arthroplasty for a 65-year-old patient with advanced osteoarthritis and fixed flexion deformity",
    "Enumerate the most important operative steps in total knee arthroplasty for a 65-year-old patient with advanced osteoarthritis and Valgus deformity.",
    "What percentage of patients with complete peroneal nerve palsy after total hip arthroplasty will never recover full strength?",
    "A 68-year-old patient presents 8 months after total knee arthroplasty with complaints of giving way while descending stairs and recurrent swelling. What is your differential diagnosis?",
    "What investigations are required to assess patellar maltracking after TKA?",
    "A 68-year-old male presents 8 months after undergoing Total knee arthroplasty. He complains of a painful catching sensation in the knee while rising from a chair. He describes a distinct clunk when extending his knee from a flexed position. What is the diagnosis. How to treat this condition?",
    "Before doing a  total knee replacement what pre operative investigations should I do especially in a patient with diabetes mellitus",
    "What is the difference between measured resection and gap balancing in total knee replacement",
    "what are the conservative treatment measures in a patient with ostearthritis of knee",
    "What is inlay vs onlay patellar implant ",
    "What is Bristol patellar wear score",
    "What is fellar patellar score?",
    "I am having a knee with extra-articular deformity of 40 degress in the tibia .I want to do a total knee replacement. What principles should I follow to proceed with TKR"

]

GROUND_TRUTHS = [
    """In a total knee replacement, inability to fully extend the knee indicates a tight extension gap, and tibial tray lift-off beyond 90 degrees of flexion indicates a tight flexion gap; therefore, both flexion and extension gaps are tight. When both gaps are equally tight, the appropriate intervention is additional proximal tibial resection because the tibial cut affects both the flexion and extension gaps equally. By recutting the proximal tibia slightly, both gaps are increased uniformly, restoring full extension, eliminating tibial tray lift-off in flexion, and achieving a balanced knee throughout the range of motion.""",

    """A 40-year-old man with groin pain and radiographic evidence of femoral head avascular necrosis showing subchondral lucency without collapse represents early (pre-collapse) disease, in which the necrotic trabecular bone becomes structurally weak and prone to subchondral fracture and eventual collapse. The progression to collapse occurs largely due to increased osteoclastic resorption during the revascularization phase, which further weakens the already compromised bone. Bisphosphonate therapy has been shown to decrease the risk of femoral head collapse in this stage by inhibiting osteoclast-mediated bone resorption, thereby preserving trabecular architecture and maintaining subchondral bone strength. By slowing excessive bone turnover and structural deterioration, bisphosphonates help delay or prevent progression to collapse and may postpone the need for surgical intervention.""",

    """The five most important operative steps in total knee arthroplasty for a 65-year-old patient with advanced osteoarthritis and fixed flexion deformity depend on the severity of the deformity. For mild deformity, the key steps are excision of medial and posterior osteophytes to remove bony blocks to extension, along with posteromedial soft tissue release to correct tightness contributing to the flexion contracture. For moderate deformity, additional steps include posterior capsular release to address persistent extension tightness, decreasing the tibial slope to improve extension balance, releasing up to the tendinous origin of the gastrocnemius when required, and performing pie-crusting of the medial collateral ligament (MCL) to correct residual medial tightness. For severe deformity, more extensive correction is needed, including an extra distal femoral cut to increase the extension gap, medial epicondylar osteotomy to facilitate adequate soft tissue balancing, and, when instability persists despite releases, the use of constrained implants to achieve a stable and well-balanced knee.""",

    """In total knee arthroplasty for a valgus knee deformity, a lateral parapatellar approach can be used to facilitate direct access to the contracted lateral structures. The tibial resection is performed in the standard manner, without alteration. For the femur, a 3 degree valgus distal femoral cut is taken to help restore appropriate alignment. Soft tissue balancing is critical and follows the Ranawat inside-out release technique, beginning with removal of lateral osteophytes, followed by sequential release of the PCL (if required), posterolateral capsule, iliotibial band, further posterolateral capsular release, and popliteus release as needed. During flexion gap balancing, the epicondylar axis is used as the primary reference for femoral component rotation, along with the cut surface of the tibia, while the posterior condylar reference is usually not relied upon because it is unreliable in valgus knees due to lateral condylar hypoplasia. An alternative technique for balancing is lateral epicondylar osteotomy when additional correction is required.""",

    """Approximately 40 to 50 percent of patients with complete peroneal nerve palsy after total hip arthroplasty will never recover full strength.""",

    """A 68-year-old patient presenting 8 months after total knee arthroplasty with complaints of giving way while descending stairs and recurrent swelling raises concern for several possibilities. The differential diagnosis includes flexion instability (especially mid-flexion or late flexion instability, which commonly presents with giving way on stairs), component malposition or malrotation leading to imbalance, polyethylene wear or early mechanical loosening, extensor mechanism insufficiency, patellofemoral instability or maltracking, periprosthetic joint infection (chronic low-grade infection presenting with recurrent effusion), and aseptic loosening of components.""",

    """Skyline (Merchant) view, CT scan (to assess femoral and tibial component rotation), Long-leg alignment films.""",

    """Diagnosis: Patellar clunk syndrome. Treatment: Initial management may include observation if symptoms are mild. Definitive treatment is arthroscopic or open debridement of the fibrous nodule at the superior pole of the patella/posterior quadriceps tendon that catches in the intercondylar box of the femoral component. This typically relieves the catching sensation and clunk.""",
    """Before total knee replacement in a patient with Diabetes Mellitus, pre-operative evaluation includes routine tests such as CBC, renal and liver function tests, serum electrolytes, coagulation profile, blood grouping, urine analysis and culture, ECG, and chest X-ray, along with joint-specific X-rays; additionally, strict assessment of glycemic control with fasting and postprandial blood sugar and HbA1c (ideally <7–8%) is essential, and screening for diabetic complications should be done including renal function (creatinine, eGFR, urine albumin), cardiovascular status (ECG ± echocardiography or stress test), and infection foci (urinary, dental, skin/foot), along with inflammatory markers like ESR and CRP, since uncontrolled diabetes and occult infections significantly increase the risk of postoperative complications such as wound infection and poor healing.""",
     """In total knee replacement, measured resection and gap balancing are two techniques used to achieve proper alignment and soft-tissue balance: in measured resection, bone cuts are made first based on anatomical landmarks (such as femoral condyles, transepicondylar axis, and tibial alignment), and the soft tissues are then adjusted secondarily to fit the prosthesis, making it more anatomy-driven but potentially less accurate in cases with deformity; in contrast, gap balancing focuses first on achieving equal and rectangular flexion and extension gaps by sequential soft-tissue releases, and bone cuts are then tailored to maintain these balanced gaps, making it more ligament-driven and often preferred in cases with severe deformity or instability, as it provides better soft-tissue balance but may alter native anatomy.""",
    """Conservative management of Osteoarthritis of the knee focuses on pain relief and functional improvement and includes lifestyle measures such as weight reduction and activity modification (avoiding squatting, prolonged standing, and high-impact activities), physiotherapy with quadriceps strengthening and range-of-motion exercises, use of assistive devices like a cane or walker, and knee braces; pharmacological options include paracetamol, NSAIDs (topical or oral), and intra-articular injections like corticosteroids or hyaluronic acid, while supportive therapies such as heat therapy, hydrotherapy (water-based exercises), sauna baths, and other forms of thermotherapy help reduce pain and stiffness, along with footwear modification and patient education as integral parts of non-surgical care.""",
    """The great majority of currently available patellar components are of the onlay type whereby the implant is placed onto the cut retropatellar surface. Inlay patellae are inserted into a reamed cavity, which, to provide adequate stability, requires the preser- vation of a certain amount of surrounding bone. Such implants are hence generally smaller than their onlay counterparts. The technique of using an inlay patella was initially described by Gschwend in 1978.176 It has been suggested that inlay patellar components provide greater composite strength between the implant and patella and may decrease the amount of patellar tilt and shift.150,253,254 However, insetting the patellar component is not without risks because overzealous removal of subchondral cancellous bone may weaken the patella, increasing its suscep- tibility to fracture. Therefore the preservation of a minimum of
15 mm of bone has been recommended to minimize surface strain on the patella.224,365 In a biomechanical analysis, Wulff and Incavo recorded increased patellar surface strains in inlay compared with onlay designs. In addition, onlay patellae were noted to be more tolerant to excess cutting. Overcutting the patella by 2 mm created a 22% increase in surface strain com- pared with a 42% increase when the patella was overreamed by the same amount.484
Critics of inlay designs have raised concerns regarding potential detrimental effects caused through the contact between remaining cartilage, bone, and soft tissues with the femoral component.214 In a cadaver study, Ezzet et al. observed similarities in patellar kinematics among implant types, although inlay components showed a higher tendency to lateral shift and tilt.135 However, some in vivo studies revealed less patellar tilt and better overall patellar alignment in patients with inlay implants.165,358 Rand and Gustilo compared 135 onlay and 116 inlay patellar components using an identical total knee system, recording a lateral release rate of 79% and 28%, respec- tively.358 Thus the authors concluded that using an inlay implant would result in better ability to centralize the extensor mecha- nism. Despite such reports, benefits of inlay patellae have remained largely theoretical and have not been converted into improved clinical outcomes
""",
    """Bristol Patella Wear Score The retropatellar surface is
divided into four zones and
graded:
0 = normal cartilage
1 = softened cartilage 2 = fibrillated/fissured
cartilage
3 = exposed bone Maximum total score 12
""",
  """ Feller Patellar Score
Anterior Knee Pain
None Mild Moderate Severe
Quadriceps Strength
Good (5/5) Fair (4/5) Poor (<4)
Feller Patellar Score138
Ability to Rise From a Chair
With ease (no arms) With ease (with arms) With difficulty
Unable
Stair Climbing
1 foot/stair, no support 1 foot/stair, with support 2 feet/stair, no support 2 feet/stair, with support Total score
15 10 5 0
5 3 1
5 3 1 0
5 4 3 2
Max. 30
""",
 """In a patient with severe extra-articular tibial deformity (~40°) undergoing total knee replacement, the key principle is to determine whether the deformity can be corrected intra-articularly or requires an extra-articular osteotomy; generally, deformities >20–30° (as in this case) cannot be safely corrected by bone cuts alone without compromising ligament balance, so a staged or simultaneous corrective osteotomy with TKR is usually indicated; careful pre-operative planning with long-leg alignment views is essential to restore the mechanical axis, and during surgery, preservation of collateral ligament attachments, achieving balanced flexion–extension gaps, and proper component positioning are critical; intramedullary guides may be unreliable in deformed tibia, so extramedullary guides or navigation/robotics are preferred, and the use of stems or constrained prostheses may be required for stability, while ensuring gradual deformity correction to avoid neurovascular complications."""   ]
print(f" Test queries ({len(TEST_QUERIES)}) and ground truths ({len(GROUND_TRUTHS)}) defined")

RUNNING EVALUATION
 Test queries (15) and ground truths (15) defined


In [ ]:
print("RUNNING EVALUATION")

if not indexer.is_built:
    print(" ERROR: Index is not built. Please run Block 33 first.")
else:
    evaluator = ComprehensiveEvaluator(experiment_logger)
    results   = evaluator.run_ablation_study(
        indexer=indexer,
        model=model,
        tokenizer=tokenizer,
        queries=TEST_QUERIES,
        ground_truths=GROUND_TRUTHS,
        hyperparameters=BEST_PARAMS,
    )
    print("\nEVALUATION COMPLETE!")
    print(f"  Results saved to : {EXPERIMENT_LOG_PATH}")
    print(f"  Cache location   : {CACHE_FOLDER}")

RUNNING EVALUATION


[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


ABLATION STUDY: RAG vs Vanilla
Queries           : 15
chunk_size        : 1000
chunk_overlap     : 100
top_k             : 10
rerank_enabled    : False
temperature       : 0.0


CONFIGURATION: RAG

Query 1/15: After insertion of the trial components in a total knee replacement, the surgeon...
   Answer preview : 1.  Assess the flexion gap: If the flexion gap is too tight, consider a thinner tibial insert or more distal femoral res...


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


...

RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


   ROUGE-1 F1     : 0.157 | BERTScore F1: 0.786 | METEOR: 0.162
   Status         : success
Query 15/15: I am having a knee with extra-articular deformity of 40 degress in the tibia .I ...
   Answer preview : 1.  Assess the patient's overall health, functional status, and expectations.
2.  Evaluate the severity and stability of...


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


   ROUGE-1 F1     : 0.300 | BERTScore F1: 0.842 | METEOR: 0.174
   Status         : success
ABLATION STUDY COMPLETE

EVALUATION COMPLETE!
  Results saved to : /content/drive/MyDrive/Knee_Arthroplasty_RAG/final_experiments_optimized.xlsx
  Cache location   : /content/drive/MyDrive/Knee_Arthroplasty_RAG/Cache
